In [1]:
"""
00_data_collection.py
=====================
BIM Sign Language — Data Collection Script

FIXES APPLIED:
  #1 — SEQUENCE_LENGTH corrected to 8 (was 20, must match training)
  #2 — MIN_FRAME_NONZERO_RATIO and MIN_SEQ_NONZERO_RATIO both standardized to 0.45
  #3 — mirror_sequence() hip swap removed (POSE_IDX only has 6 points = 18 values,
       indices 18–24 don't exist — the old code was silently swapping zeros)
  #4 — DATA_PATH changed to "dataset_phone" to match train.py
  #5 — Mirror saving left commented out but clearly documented
"""

import cv2
import mediapipe as mp
import numpy as np
import os
import time

# ==============================
# CONFIG — single source of truth
# Keep these in sync with:
#   02_train_model.py
#   03_test_model_desktop.py
#   04_fastapi_server.py
#   05_camera_feature_extraction.dart
# ==============================
DATA_PATH       = "dataset_phone"   # FIX #4: was "dataset", must match train.py
SEQUENCE_LENGTH = 8                 # FIX #1: was 20, must match training

BASE_TARGET_SEQUENCES   = 40        # static mode: sequences per round
COUNTDOWN_SECONDS       = 3         # dynamic mode: countdown before recording

HAND_LANDMARKS = 21
HAND_VEC_LEN   = HAND_LANDMARKS * 3   # 63 per hand
HAND_FEATURES  = HAND_VEC_LEN * 2     # 126 both hands

# Face indices — must match training + testing + Flutter
SELECTED_FACE_IDX = [
    13, 14,
    78, 308, 82, 312,
    33, 133,
    362, 263,
    70, 63, 105, 66, 107,
    336, 296, 334, 293, 300
]
FACE_FEATURES = len(SELECTED_FACE_IDX) * 3   # 20 * 3 = 60

# Pose: shoulders, elbows, wrists only (6 points × 3 = 18)
# NOTE: No hips — POSE_FEATURES is exactly 18 values (indices 0–17)
POSE_IDX = [
    11, 13, 15,   # left shoulder, elbow, wrist
    12, 14, 16,   # right shoulder, elbow, wrist
]
POSE_FEATURES = len(POSE_IDX) * 3   # 6 * 3 = 18

FEATURE_SIZE = HAND_FEATURES + FACE_FEATURES + POSE_FEATURES  # 126+60+18 = 204

# FIX #2: standardized to 0.45 (was 0.5 frame / 0.6 sequence — inconsistent)
MIN_FRAME_NONZERO_RATIO = 0.45
MIN_SEQ_NONZERO_RATIO   = 0.45


# ==============================
# MIRROR AUGMENTATION
# FIX #3: Removed hip swap — pose block is only 18 values (0:18).
#         The old code tried to access indices 18:24 which don't exist,
#         so the hip swap was silently doing nothing.
#         Now correctly swaps only left arm (0:9) ↔ right arm (9:18).
# ==============================
def mirror_sequence(sequence):
    """
    Horizontal mirror augmentation:
    - Flip x coordinates (x' = 1 - x) for all features
    - Swap left hand ↔ right hand blocks
    - Swap left arm ↔ right arm in pose block
    - Face points flipped in x only (indices unchanged)
    """
    seq_m = sequence.copy()

    for t in range(seq_m.shape[0]):
        frame = seq_m[t]

        # 1. Flip x for every feature (x is at every 3rd index starting at 0)
        for i in range(0, FEATURE_SIZE, 3):
            frame[i] = 1.0 - frame[i]

        # 2. Swap left hand ↔ right hand
        left_hand  = frame[0:HAND_VEC_LEN].copy()
        right_hand = frame[HAND_VEC_LEN:HAND_FEATURES].copy()
        frame[0:HAND_VEC_LEN]          = right_hand
        frame[HAND_VEC_LEN:HAND_FEATURES] = left_hand

        # 3. Swap left arm ↔ right arm in pose block
        # FIX #3: pose_block is 18 values (0:18 only — no hips)
        # left arm  = indices 0:9  (shoulder+elbow+wrist × 3)
        # right arm = indices 9:18
        pose_start = HAND_FEATURES + FACE_FEATURES
        pose_block = frame[pose_start : pose_start + POSE_FEATURES].copy()

        left_arm  = pose_block[0:9].copy()
        right_arm = pose_block[9:18].copy()

        pose_block[0:9]  = right_arm
        pose_block[9:18] = left_arm

        frame[pose_start : pose_start + POSE_FEATURES] = pose_block

    return seq_m


# ==============================
# MEDIAPIPE
# ==============================
mp_holistic = mp.solutions.holistic
mp_drawing  = mp.solutions.drawing_utils


# ==============================
# MODE & CATEGORY / LABEL SETUP
# ==============================
print("=" * 45)
print("  BIM Sign Language — Data Collection")
print("=" * 45)
print("\nSelect mode:")
print("  1 — Static sign  (hold the pose)")
print("  2 — Dynamic sign (moving gesture)")
mode       = input("Enter 1 or 2: ").strip()
is_dynamic = (mode == "2")

# Frame skipping: dynamic signs need to span more time
# FRAME_SKIP=4 at 30fps → each accepted frame is ~133ms apart
# 8 frames × 133ms ≈ 1 second of motion captured per sequence
FRAME_SKIP = 4 if is_dynamic else 1

os.makedirs(DATA_PATH, exist_ok=True)

# ---- Category selection ----
existing_categories = sorted([
    d for d in os.listdir(DATA_PATH)
    if os.path.isdir(os.path.join(DATA_PATH, d))
])

if existing_categories:
    print("\nExisting categories:")
    for i, cat in enumerate(existing_categories, start=1):
        print(f"  {i} — {cat}")
    print("  0 — Create new category")
    choice = input("Select category number, or 0 for new: ").strip()
    if choice.isdigit() and 1 <= int(choice) <= len(existing_categories):
        category = existing_categories[int(choice) - 1]
        print(f"Using existing category: {category}")
    else:
        category = input("Enter new category name: ").strip()
        print(f"Creating new category: {category}")
else:
    category = input("No categories yet. Enter new category name: ").strip()
    print(f"Creating first category: {category}")

category_path = os.path.join(DATA_PATH, category)
os.makedirs(category_path, exist_ok=True)

# ---- Label (sign name) ----
print(f"\nCategory: {category}")
label      = input("Enter sign label: ").strip()
label_path = os.path.join(category_path, label)
os.makedirs(label_path, exist_ok=True)

# Resume counter from existing files
existing_files = [f for f in os.listdir(label_path) if f.endswith(".npy")]
if existing_files:
    counters = [
        int(f.split("_")[-1].split(".")[0])
        for f in existing_files
        if f.split("_")[-1].split(".")[0].isdigit()
    ]
    counter = (max(counters) + 1) if counters else 0
else:
    counter = 0

target_sequences = counter + BASE_TARGET_SEQUENCES if not is_dynamic else None

print(f"\n✅ Setup complete")
print(f"   Mode     : {'Dynamic' if is_dynamic else 'Static'}")
print(f"   Category : {category}")
print(f"   Label    : {label}")
print(f"   Seq len  : {SEQUENCE_LENGTH} frames")
print(f"   Frame skip: {FRAME_SKIP}")
print(f"   Starting from sequence #{counter}")
if not is_dynamic:
    print(f"   Target   : {target_sequences} sequences")
print("\nPress SPACE to begin. Press ESC to quit.")


# ==============================
# CAPTURE LOOP
# ==============================
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("❌ Error: Could not open webcam.")
    exit(1)

sequence      = []
prev_time     = 0
started       = False
waiting_for_space = False
frame_counter = 0

# Dynamic mode state machine
state                = "IDLE"
countdown_start_time = None

with mp_holistic.Holistic(
    static_image_mode=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_counter += 1
        frame = cv2.flip(frame, 1)

        curr_time = time.time()
        fps       = 1 / (curr_time - prev_time) if prev_time else 0
        prev_time = curr_time

        keypoints = np.zeros(FEATURE_SIZE, dtype=np.float32)

        # ─── KEYPOINT EXTRACTION ───────────────────────────────────────────
        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)

        # Hands
        for h_idx, hand in enumerate([results.left_hand_landmarks,
                                       results.right_hand_landmarks]):
            if hand:
                base = h_idx * HAND_VEC_LEN
                for i, lm in enumerate(hand.landmark):
                    keypoints[base + i * 3]     = lm.x
                    keypoints[base + i * 3 + 1] = lm.y
                    keypoints[base + i * 3 + 2] = lm.z
                mp_drawing.draw_landmarks(
                    frame, hand, mp_holistic.HAND_CONNECTIONS)

        # Face
        if results.face_landmarks:
            h, w, _ = frame.shape
            for i, idx in enumerate(SELECTED_FACE_IDX):
                lm     = results.face_landmarks.landmark[idx]
                offset = HAND_FEATURES + i * 3
                keypoints[offset]     = lm.x
                keypoints[offset + 1] = lm.y
                keypoints[offset + 2] = lm.z
                cv2.circle(frame, (int(lm.x * w), int(lm.y * h)),
                           2, (0, 255, 0), -1)

        # Pose
        if results.pose_landmarks:
            h, w, _ = frame.shape
            print("Frame size:", frame.shape)
            pose_start = HAND_FEATURES + FACE_FEATURES
            for i, idx in enumerate(POSE_IDX):
                lm     = results.pose_landmarks.landmark[idx]
                offset = pose_start + i * 3
                keypoints[offset]     = lm.x
                keypoints[offset + 1] = lm.y
                keypoints[offset + 2] = lm.z
                cv2.circle(frame, (int(lm.x * w), int(lm.y * h)),
                           3, (255, 0, 0), -1)

        non_zero_ratio = np.count_nonzero(keypoints) / keypoints.size

        # ─── HELPER: draw common UI info ───────────────────────────────────
        def draw_info(extra_y=0):
            cv2.putText(frame,
                        f"{'Dynamic' if is_dynamic else 'Static'} | {category} / {label}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65,
                        (255, 255, 255), 2)
            cv2.putText(frame, f"FPS: {int(fps)}",
                        (10, 55 + extra_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 200, 0), 1)
            cv2.putText(frame, "ESC to exit",
                        (10, 75 + extra_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        # ══════════════════════════════════════════════════════════════════
        # STATIC MODE
        # ══════════════════════════════════════════════════════════════════
        if not is_dynamic:

            # Wait for first SPACE
            if not started:
                cv2.putText(frame, "Press SPACE to start collecting",
                            (10, 100), cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, (0, 255, 255), 2)
                draw_info()
                cv2.imshow("BIM Data Collection", frame)
                key = cv2.waitKey(1)
                if key == 32:
                    started = True
                elif key == 27:
                    break
                continue

            # Pause between rounds
            if waiting_for_space:
                cv2.putText(frame,
                            f"Done {counter} seq! SPACE=more  ESC=exit",
                            (10, 100), cv2.FONT_HERSHEY_SIMPLEX,
                            0.65, (0, 255, 0), 2)
                draw_info()
                cv2.imshow("BIM Data Collection", frame)
                key = cv2.waitKey(1)
                if key == 32:
                    target_sequences += BASE_TARGET_SEQUENCES
                    waiting_for_space  = False
                elif key == 27:
                    break
                continue

            # Collect frames
            if non_zero_ratio < MIN_FRAME_NONZERO_RATIO:
                cv2.putText(frame, "Low keypoint quality — adjust position",
                            (10, 200), cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (0, 0, 255), 2)
            else:
                if frame_counter % FRAME_SKIP == 0:
                    sequence.append(keypoints)

            # Sequence complete
            if len(sequence) == SEQUENCE_LENGTH:
                seq_array = np.array(sequence, dtype=np.float32)
                sequence  = []

                seq_nz = np.count_nonzero(seq_array) / seq_array.size
                if seq_nz < MIN_SEQ_NONZERO_RATIO:
                    print(f"  ⚠️  Discarding sequence #{counter} "
                          f"(quality {seq_nz:.0%} < {MIN_SEQ_NONZERO_RATIO:.0%})")
                else:
                    path = os.path.join(label_path, f"{label}_{counter}.npy")
                    np.save(path, seq_array)
                    print(f"  ✅ [STATIC] Saved sequence #{counter}  "
                          f"shape={seq_array.shape}")
                    counter += 1
                    if counter >= target_sequences:
                        waiting_for_space = True

            # UI
            cv2.putText(frame,
                        f"Sequences: {counter}/{target_sequences}",
                        (10, 100), cv2.FONT_HERSHEY_SIMPLEX,
                        0.65, (0, 255, 0), 2)
            cv2.putText(frame,
                        f"Frames buffered: {len(sequence)}/{SEQUENCE_LENGTH}",
                        (10, 125), cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0, 255, 255), 2)

            # Progress bar
            h_frame = frame.shape[0]
            bar_w   = int((len(sequence) / SEQUENCE_LENGTH) * 200)
            cv2.rectangle(frame, (10, h_frame - 30),
                          (210, h_frame - 15), (80, 80, 80), -1)
            cv2.rectangle(frame, (10, h_frame - 30),
                          (10 + bar_w, h_frame - 15), (0, 200, 255), -1)

            draw_info(extra_y=80)
            cv2.imshow("BIM Data Collection", frame)
            if cv2.waitKey(1) == 27:
                break

        # ══════════════════════════════════════════════════════════════════
        # DYNAMIC MODE
        # ══════════════════════════════════════════════════════════════════
        else:
            key = cv2.waitKey(1)

            # ── IDLE: waiting for SPACE ────────────────────────────────────
            if state == "IDLE":
                sequence = []
                cv2.putText(frame,
                            "SPACE to record one sequence | ESC to exit",
                            (10, 100), cv2.FONT_HERSHEY_SIMPLEX,
                            0.65, (0, 255, 255), 2)
                cv2.putText(frame,
                            f"Sequences saved: {counter}",
                            (10, 125), cv2.FONT_HERSHEY_SIMPLEX,
                            0.65, (0, 255, 0), 2)
                draw_info()
                cv2.imshow("BIM Data Collection", frame)

                if key == 32:
                    state                = "COUNTDOWN"
                    countdown_start_time = curr_time
                elif key == 27:
                    break
                continue

            # ── COUNTDOWN ─────────────────────────────────────────────────
            elif state == "COUNTDOWN":
                elapsed   = curr_time - countdown_start_time
                remaining = int(max(0, COUNTDOWN_SECONDS - elapsed)) + 1
                cv2.putText(frame, f"Get ready: {remaining}",
                            (10, 100), cv2.FONT_HERSHEY_SIMPLEX,
                            1.5, (0, 255, 255), 3)
                cv2.putText(frame,
                            "Perform the full sign after countdown",
                            (10, 145), cv2.FONT_HERSHEY_SIMPLEX,
                            0.65, (0, 255, 0), 2)
                draw_info()
                cv2.imshow("BIM Data Collection", frame)

                if elapsed >= COUNTDOWN_SECONDS:
                    sequence = []
                    state    = "RECORDING"
                if key == 27:
                    break
                continue

            # ── RECORDING ─────────────────────────────────────────────────
            elif state == "RECORDING":
                cv2.putText(frame, "● REC",
                            (10, 55), cv2.FONT_HERSHEY_SIMPLEX,
                            1.0, (0, 0, 255), 3)
                cv2.putText(frame,
                            f"Frames: {len(sequence)}/{SEQUENCE_LENGTH}",
                            (10, 90), cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, (0, 255, 255), 2)
                cv2.putText(frame,
                            f"Seq #: {counter}",
                            (10, 115), cv2.FONT_HERSHEY_SIMPLEX,
                            0.65, (0, 255, 0), 2)

                # Progress bar
                h_frame = frame.shape[0]
                bar_w   = int((len(sequence) / SEQUENCE_LENGTH) * 200)
                cv2.rectangle(frame, (10, h_frame - 30),
                              (210, h_frame - 15), (80, 80, 80), -1)
                cv2.rectangle(frame, (10, h_frame - 30),
                              (10 + bar_w, h_frame - 15), (0, 0, 255), -1)

                if non_zero_ratio >= MIN_FRAME_NONZERO_RATIO:
                    if frame_counter % FRAME_SKIP == 0:
                        sequence.append(keypoints)
                else:
                    cv2.putText(frame,
                                "Low keypoint quality — keep hands visible",
                                (10, 145), cv2.FONT_HERSHEY_SIMPLEX,
                                0.6, (0, 0, 255), 2)

                draw_info(extra_y=80)
                cv2.imshow("BIM Data Collection", frame)

                # Sequence complete
                if len(sequence) >= SEQUENCE_LENGTH:
                    seq_array = np.array(
                        sequence[:SEQUENCE_LENGTH], dtype=np.float32)

                    seq_nz = np.count_nonzero(seq_array) / seq_array.size
                    if seq_nz < MIN_SEQ_NONZERO_RATIO:
                        print(f"  ⚠️  Discarding dynamic sequence #{counter} "
                              f"(quality {seq_nz:.0%})")
                    else:
                        # Save original
                        orig_path = os.path.join(
                            label_path, f"{label}_{counter}.npy")
                        np.save(orig_path, seq_array)
                        print(f"  ✅ [DYNAMIC] Saved sequence #{counter}  "
                              f"shape={seq_array.shape}")

                        # FIX #5: Mirror augmentation left commented out.
                        # Uncomment ONLY if you want offline mirror augmentation.
                        # Do NOT uncomment if using augmentation in train.py
                        # (spatial_augment already handles mirroring variation).
                        # mirror_path = os.path.join(
                        #     label_path, f"{label}_{counter}_mirror.npy")
                        # np.save(mirror_path, mirror_sequence(seq_array))

                        counter += 1

                    state = "IDLE"

                if key == 27:
                    break
                continue

cap.release()
cv2.destroyAllWindows()
print(f"\n✅ Data collection complete — {counter} sequences saved to {label_path}")

  BIM Sign Language — Data Collection

Select mode:
  1 — Static sign  (hold the pose)
  2 — Dynamic sign (moving gesture)

Existing categories:
  1 — Alphabet
  0 — Create new category
Using existing category: Alphabet

Category: Alphabet

✅ Setup complete
   Mode     : Static
   Category : Alphabet
   Label    : J
   Seq len  : 8 frames
   Frame skip: 1
   Starting from sequence #0
   Target   : 40 sequences

Press SPACE to begin. Press ESC to quit.
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Frame size: (720, 1280, 3)
Fram